# **ColabSeqDisplay · Predict**

<img src="https://img.shields.io/badge/Paper-not%20yet%20posted-lightgrey" style="max-width: 100%;">
<a href="https://github.com/JasonJiangs/ColabSeqDisplay"><img src="https://img.shields.io/badge/Github-black?logo=github" style="max-width: 100%;"></a>
<a href="https://colab.research.google.com/github/JasonJiangs/ColabSeqDisplay/blob/main/colab/ColabSeqDisplay_Predict.ipynb"><img src="https://img.shields.io/badge/Open%20in-Colab-F9AB00?logo=googlecolab&logoColor=white" style="max-width: 100%;"></a>
<a href="https://github.com/JasonJiangs/ColabSeqDisplay"><img src="https://img.shields.io/badge/License-see%20repository-lightgrey" style="max-width: 100%;"></a>

- **You have a `model_bundle.zip` and a list of variants you have not made yet.** This notebook scores them and hands back a ranked table. That is all it does.

- The bundle carries everything the model needs — the backbone name, the LoRA weights, the head, the library spec and the frozen coordinates of the mutated sites — so the variants you type here are built against the same wild type, at the same positions, as the ones it was trained on. A bundle from a different library is refused rather than quietly mis-scored, and it needs no 3Di string: the one its run used is frozen inside it.

- **Not the training notebook.** There is nothing here about choosing a backbone, hyperparameters or preparation.

- Nothing is uploaded anywhere. The predictions come back as a CSV you download.

- **Two notebooks.** [ColabSeqDisplay](https://colab.research.google.com/github/JasonJiangs/ColabSeqDisplay/blob/main/colab/ColabSeqDisplay.ipynb) trains a model on your library and exports it. [ColabSeqDisplay_Predict](https://colab.research.google.com/github/JasonJiangs/ColabSeqDisplay/blob/main/colab/ColabSeqDisplay_Predict.ipynb) scores new variants from a saved `.zip`.

# How to start

## 1 · Switch this runtime to a GPU

`Runtime` ▸ `Change runtime type` ▸ **T4 GPU** ▸ `Save`. Colab restarts the runtime and clears anything you had already run.

## 2 · Click the run-button

Hover over the cell below and click ▶ on its left. It installs the package (1–2 minutes the first time), then draws the panel that **is** this notebook. There is no code to write and no other cell to edit.

## 3 · Which GPU

- **T4 — <font color="red">free</font>.** 16 GB, enough for most of this. <font color="red">Free sessions are pre-empted and capped in length, so a run measured in hours is a run you will probably lose. Mount Drive first — see below.</font>
- **L4 (needs Colab Pro).** 24 GB — the smallest card that runs the three backbones that will not fit a T4 (`SaProt-1.3B`, `ProtT5-XL`, `Ankh-large`), and a steadier session.
- **A100 (needs Colab Pro).** 40 GB and much faster; the card for a full multi-seed evaluation.
- **No GPU (CPU runtime).** Scoring still runs, slowly — see below. This is the one notebook of the two that is genuinely usable without a GPU.

### The free tier will disconnect. Mount Drive before it does.

`Files` — the folder icon in the left margin — then **Mount Drive**. Everything outside `/content/drive/MyDrive/` is thrown away when the session ends, including the backbone download and anything the panel wrote, so copy what you want to keep into Drive as it appears — the predictions CSV, and `model_bundle.zip` if the only copy of it is the one you uploaded into this runtime.

### Scoring is much cheaper than training

Prediction is one forward pass per variant with no gradients, so the guide above is stricter than this notebook needs: a free T4 runs the 650M-parameter backbones that are painful to *train* there.

The exception is a bundle naming one of `SaProt-1.3B`, `ProtT5-XL`, `Ankh-large`: the registry has no free-T4 figure for any of them, so scoring one may still exhaust a T4 and nothing here can price that run in advance — move to an L4 or A100 if it runs out of memory. And a screen of tens of thousands of variants is still a long job: the panel costs it out before it starts. Read that estimate before you press the button.

**Without a GPU this notebook still works**, on the CPU: fine for a few dozen designs you want to rank before ordering, painful for a library-sized screen.

### A bundle from the older, longer list still scores

**This notebook does not check your bundle against the training notebook's list of backbones.** It reads the backbone name out of the bundle, and a model trained on any of the six backbones the training form no longer offers but this package still builds — `ProtT5-XL`, `Ankh-large`, `ESMC-300M`, `ESMC-600M`, `SeqDance`, `ESMDance` — loads and scores as it always did; the panel adds a note saying why that backbone is not offered for *training* any more.

Three of them — `ProtT5-XL`, `ESMC-300M`, `ESMC-600M` — need one package this notebook does not install. Nothing is lost: the loader stops and names the package, and `pip install` it plus a runtime restart is the whole fix.

The one refusal is a bundle naming `METL`, which has no adapter in this package. That is said when the bundle is loaded, not halfway through a screen.

In [ ]:
#@title **Click the run-button to score variants** { display-mode: "form" }

#@markdown ### Hint
#@markdown - **A file picker freezes this page while it is open.** Every control stops responding until you pick a file or press **Cancel upload**. The panel names the file it is waiting for before the dialog opens.
#@markdown - **The ▶ button.** It spins while the cell installs the package and builds the panel, then goes back to ▶ — that means finished, not broken. The panel stays live after the cell ends; if it stops responding, click ▶ again to rebuild it.
#@markdown ### <font color=red>If the session disconnects</font>
#@markdown - <font color=red>Reconnect and run this cell again: the panel comes back, but the runtime is empty. Whatever was under `/content` is gone; whatever you wrote to a mounted Google Drive folder is not. Mount Drive before you start anything long.</font>
#@markdown - <font color=red>Changing the runtime type empties it the same way. Stop this cell first, change the runtime, then run it again.</font>
#@markdown ### Where the code comes from
#@markdown - The field below names what gets installed — one repository, fine-tuning engine included. A **folder path** works as well as a URL, and is installed with `pip install -e`.
colabsd_repository = "https://github.com/JasonJiangs/ColabSeqDisplay.git"  #@param {type:"string"}

import importlib
import importlib.util
import subprocess
import sys
from pathlib import Path

WORK_ROOT = Path.cwd()


def run_command(command):
    """Run a command, raising with its own output when it fails."""
    parts = [str(part) for part in command]
    finished = subprocess.run(parts, capture_output=True, text=True)
    if finished.returncode != 0:
        raise RuntimeError(
            "This command failed:\n  " + " ".join(parts) + "\n"
            + (finished.stdout or "")[-1500:] + (finished.stderr or "")[-1500:]
        )
    return finished


def head_of(repo):
    """The commit a checkout is on, or "" when it is not a git repository."""
    try:
        return run_command(["git", "-C", str(repo), "rev-parse", "HEAD"]).strip()
    except RuntimeError:
        return ""


def checkout(source, name):
    """A local folder as given, or a clone of a git URL beside this notebook, brought up to date.

    An existing clone is fetched and reset onto the remote rather than left alone. Colab keeps a
    runtime alive across many hours: without this, a fix published after the first run of the
    session can never reach the user, because the package is already installed and the old
    fast path did nothing at all.
    """
    local = Path(source).expanduser()
    if local.is_dir():
        return local.resolve(), False
    target = WORK_ROOT / name
    if not (target / ".git").is_dir():
        print("cloning " + str(source) + " ...")
        try:
            run_command(["git", "clone", "--depth", "1", source, target])
        except RuntimeError as exc:
            raise RuntimeError(
                str(exc) + "\n\n" + name + " could not be downloaded from " + str(source) + ". In the "
                "field at the top of this form, put a repository this runtime can reach, or the path of a "
                "folder you uploaded to it (for example " + str(target) + ")."
            ) from None
        return target.resolve(), True

    before = head_of(target)
    try:
        run_command(["git", "-C", str(target), "fetch", "--depth", "1", "origin"])
        run_command(["git", "-C", str(target), "reset", "--hard", "FETCH_HEAD"])
    except RuntimeError:
        print("could not check " + str(source) + " for updates; using the copy already here")
        return target.resolve(), False
    moved = head_of(target) != before
    if moved:
        print("updated " + name + " to " + head_of(target)[:7])
    return target.resolve(), moved


def package_dir(module):
    """The directory an importable package sits in, or None when it is not importable."""
    found = importlib.util.find_spec(module)
    if found is None or not found.origin:
        return None
    return Path(found.origin).resolve().parent


def colabsd_is_complete():
    """True when colabsd is importable *and* its config registry and bundled example came with it.

    Two layouts are both correct: an editable install leaves `config/` and `examples/` beside the
    package, a built wheel carries them inside it. Either answer counts; neither does.
    """
    package = package_dir("colabsd")
    if package is None:
        return False
    return any(
        (root / "config" / "best").is_dir() and (root / "examples").is_dir()
        for root in (package, package.parent)
    )


package_root, moved = checkout(colabsd_repository, "ColabSeqDisplay")
if moved or not colabsd_is_complete():
    print("installing ColabSeqDisplay from " + str(package_root) + " ...")
    run_command([sys.executable, "-m", "pip", "install", "-q", "-e", package_root])
    importlib.invalidate_caches()
    if str(package_root) not in sys.path:
        sys.path.insert(0, str(package_root))
    for module in [name for name in sys.modules if name == "colabsd" or name.startswith("colabsd.")]:
        del sys.modules[module]

import colabsd
from colabsd.ui import core, predict_workflow

runtime = core.detect_runtime()
WORK_DIR = WORK_ROOT / "colabsd_work"

if runtime.has_gpu:
    GPU_DESCRIPTION = str(runtime.gpu_name) + "  (" + format(runtime.gpu_memory_gb or 0.0, ".1f") + " GB)"
else:
    GPU_DESCRIPTION = "none — Runtime > Change runtime type > T4 GPU, then run this cell again"

print("colabsd " + colabsd.__version__ + "   from " + str(Path(colabsd.REPO_ROOT)))
print("GPU       " + GPU_DESCRIPTION)
print("files     " + str(WORK_DIR))
print("")

wizard = predict_workflow.launch(work_dir=WORK_DIR, has_gpu=runtime.has_gpu)